# Exercise 2 · Where and when the satellite passes, and what the pixels say
**ICAT3370 Foundations of Space Data and EO Systems** · 6 October 2026

**In Exercise 1** we chose Sentinel-2 and picked one image of Vaasa from 21 June 2026. Two things remained unexplained, the orbit number and the acquisition time of around 10:10. Today we find out what they mean, download the data ourselves and read the first measurements.

**Contents**
1. Define an area of interest (AOI) for Vaasa, search the open catalogue for every Sentinel-2 image of it in one summer and find the orbit in the results
2. Download one image, clip it to Vaasa, and then read and visualize it
3. Read spectral signatures and learn how to check the quality of individual pixels

**Instructions.** Run a cell with **Shift + Enter**. You do not need to write anything in this notebook. The questions in the Discussion boxes are for discussion, first with the person next to you and then with the whole room.

***
## 0 · Did you notice?

Every Sentinel-2 image of Vaasa that we used in Exercise 1 was taken at about 10:10 UTC.

:::{admonition} Discussion
How many days do you expect between two Sentinel-2 images of Vaasa?
:::


***
## 1 · The area and the catalogue

### The area of interest

An area of interest (AOI) is defined by a polygon feature that represents a specific geographic extent. In GIS the usual way is a **polygon** stored in a **shapefile**. A shapefile is a set of files with the same name, `.shp` for the shapes, `.dbf` for the table, `.shx` as an index and `.prj` for the coordinate system. Such files are published as open data. For Finland two sources are worth knowing.

- The **National Land Survey of Finland** publishes the administrative divisions, which are the municipalities, the regions and the national border, as shapefiles and GeoPackages. See the [file service](https://asiointi.maanmittauslaitos.fi/karttapaikka/tiedostopalvelu) and the [product description](https://nic.funet.fi/index/geodata/mml/hallintorajat_10k/2025/tietotuoteseloste_kuntajako_2025_10k.pdf). CSC keeps a copy of the 2025 files that anyone can download without a login at [nic.funet.fi](https://nic.funet.fi/index/geodata/mml/hallintorajat_10k/2025/).
- **Statistics Finland** publishes statistical areas, among them the municipalities, as spatial data. See the [spatial data page](https://stat.fi/fi/palvelut/tilastodatapalvelut/paikkatietoaineistot).

While working with an area that has a water body one needs to be careful. Many boundary files are cut along the coastline or only include the land. Luckily The National Land Survey files are not. The next cell downloads the municipalities of Finland from the CSC copy. It fetches the four files of the shapefile.

:::{figure} https://gisrsstudy.com/wp-content/uploads/2020/12/shapefiles.jpg
:alt: The files that make up a shapefile
:width: 30%
:align: center

Image from [GIS and RS Study](https://gisrsstudy.com/).
:::

In [ ]:
import os
import urllib.request

base = "https://nic.funet.fi/index/geodata/mml/hallintorajat_10k/2025/SuomenKuntajako_2025_10k"
os.makedirs("data/aoi", exist_ok=True)

for ext in ["shp", "shx", "dbf", "prj"]:
    urllib.request.urlretrieve(f"{base}.{ext}", f"data/aoi/SuomenKuntajako_2025_10k.{ext}")

for file in os.listdir("data/aoi"):
    print(file, round(os.path.getsize("data/aoi/" + file) / 1e6, 2), "MB")

:::{admonition} Discussion
Which of the four files is the largest, and why? What do you expect to find inside?
:::


In [ ]:
import geopandas as gpd

municipalities = gpd.read_file("data/aoi/SuomenKuntajako_2025_10k.shp")
print(municipalities.crs)
print(len(municipalities))
municipalities.drop(columns="geometry").head()

:::{admonition} Discussion
How many municipalities does the file hold, and in which coordinate system? Besides the names, what does the table contain, and what could the three area columns be for?
:::

In [ ]:
vaasa = municipalities[municipalities["NAMEFIN"] == "Vaasa"]
vaasa.drop(columns="geometry")

The next cell draws the Vaasa polygon from the National Land Survey file.

In [ ]:
vaasa.plot()

:::{admonition} Discussion
The polygon reaches far out over the water to the west. Why does a municipal border run through the sea? What would a map of the bay in front of the city miss if the polygon stopped at the coast?
:::


### Searching the catalogue

Every Sentinel-2 product is listed in an open catalogue as soon as it is published. Such catalogues follow a common standard called **STAC**, the SpatioTemporal Asset Catalog. In STAC every product is an *item* with a time, an area, a set of properties and links to its files. A STAC *API* lets a program ask the catalogue a question, and for this one we need no account. We use the catalogue run by Element 84 on Amazon Web Services, called [Earth Search](https://earth-search.aws.element84.com/v1), and the Python package `pystac-client` ([documentation](https://pystac-client.readthedocs.io/en/stable/quickstart.html)).

A question to the catalogue has three parts, what kind of product, which area and which dates. The area is our Vaasa polygon. The catalogue works in longitude and latitude, so the polygon is converted first. The line with `make_valid` repairs a small self intersection in the Vaasa polygon, because the catalogue refuses shapes that cross themselves.

In [ ]:
from pystac_client import Client

aoi = vaasa.to_crs(4326).make_valid()

catalog = Client.open("https://earth-search.aws.element84.com/v1")

search = catalog.search(
    collections=["sentinel-2-l2a"],
    intersects=aoi.iloc[0],
    datetime="2026-06-01/2026-08-31",
)
items = list(search.items())
print(len(items))

:::{admonition} Discussion
We converted the polygon from the national grid to longitude and latitude. Why was that needed? What would you change in the cell above to ask about another place, another season or another level?
:::


Each item is one product. First one shown below.

In [ ]:
item = items[0]
print(item.id)
print(item.datetime)
item.properties

:::{admonition} Discussion
Which of these fields looks familiar to you and which are new?
:::


The next cell coverts fields into DataFrame. The product name is split at each underscore. The first part is the satellite, the fifth is the relative orbit and the sixth is the tile. Similar to our previous exercise.

In [ ]:
import pandas as pd

rows = []
for item in items:
    name = item.properties["s2:product_uri"].split("_")
    rows.append({
        "time": item.datetime,
        "satellite": name[0],
        "orbit": name[4],
        "tile": name[5],
        "cloud_percent": item.properties["eo:cloud_cover"],
        "sun_elevation": item.properties["view:sun_elevation"],
    })

scenes = pd.DataFrame(rows).sort_values("time", ignore_index=True)
scenes.head(12)

:::{admonition} Discussion
Look at the first twelve rows. What repeats and what changes from row to row? How many rows do you think the whole table has, and how many different dates?
:::

In [ ]:
print(len(scenes))
print(scenes["time"].dt.date.nunique())
print(scenes["satellite"].unique())
print(scenes["orbit"].unique())
print(scenes["tile"].unique())

:::{admonition} Discussion
The table has far more rows than dates. Where do the extra rows come from, and which column would you check to find out?
:::


The next cell draws the footprints of the tiles from one pass, the Sentinel-2B pass on 21 June.

In [ ]:
one_pass = [item for item in items
            if item.id.startswith("S2B") and item.datetime.date().isoformat() == "2026-06-21"]
footprints = gpd.GeoDataFrame.from_features([item.to_dict() for item in one_pass], crs=4326)

ax = footprints.plot(facecolor="none", edgecolor="grey")
aoi.plot(ax=ax, facecolor="none", edgecolor="red")

:::{admonition} Discussion
How many footprints are there, and how do they relate to the red area? What does that say about how big a tile is?
:::


### The time of day

The first line of the next cell counts the minutes after 10:00 UTC for every product. The plot then shows every product as one dot, the date along the bottom and the minutes up the side. Each satellite gets its own colour.

In [ ]:
import matplotlib.pyplot as plt

scenes["minute"] = (scenes["time"].dt.hour - 10) * 60 + scenes["time"].dt.minute + scenes["time"].dt.second / 60

for name, group in scenes.groupby("satellite"):
    plt.scatter(group["time"], group["minute"], label=name)
plt.ylabel("minutes after 10:00 UTC")
plt.ylim(0, 30)
plt.legend(loc="upper left", ncol=3)
plt.xticks(rotation=45)
plt.show()

The second plot is the same, with one colour for each relative orbit.

In [ ]:
for name, group in scenes.groupby("orbit"):
    plt.scatter(group["time"], group["minute"], label=name)
plt.ylabel("minutes after 10:00 UTC")
plt.ylim(0, 30)
plt.legend(loc="upper left", ncol=3)
plt.xticks(rotation=45)
plt.show()

:::{admonition} Discussion
What do you notice in the first plot? What changed in the second, and what does that say about who decides the time of day?
:::


### How often does it pass?

The next cell lists each date with a pass once, sorts the dates, and counts the days from one date to the next, whichever satellite and orbit.

In [ ]:
dates = scenes["time"].dt.normalize().drop_duplicates().sort_values()
gaps = dates.diff().dt.days.dropna().astype(int)

print(gaps.value_counts().sort_index())
print("Mean gap in days", round(gaps.mean(), 2))

:::{admonition} Discussion
Compare the mean with the number you said at the start and with the 5 days quoted for Sentinel-2. Why does the real catalogue show passes so much more often?
:::


***
## 2 · Getting the data

The search gave us a list of products. Now we take one of them, the Sentinel-2B pass of 21 June, which is the product from Exercise 1. Its name is the name you learned to read in Exercise 1, and that name is all we need to find the official files in the Copernicus Data Space, where ESA keeps them. The next cell picks the product from our search results and prints the name.

In [ ]:
product = next(item for item in items if item.id == "S2B_34VER_20260621_0_L2A")
name = product.properties["s2:product_uri"]
print(name)

:::{admonition} Discussion
Which parts of the name do you recognise from Exercise 1? What does the ending `.SAFE` tell you about what we are going to download?
:::


### Downloading the data

The official product is a **SAFE** folder, delivered as one zip file of 0.94 GB. To download it you need a account at the [Copernicus Data Space](https://dataspace.copernicus.eu/).

A download of that size takes a few minutes, and 35 downloads at the same moment would slow the network for everyone. So the product for today was downloaded beforehand and is waiting in the course project space on CSC. We do not run the download today. You will use it in the assignment.

:::{warning}
You fill in your own e-mail address in the code. The notebook asks for the password when you run it.
:::

:::{admonition} The code that downloaded the product
:class: dropdown

```python
import getpass
import requests
import zipfile

username = "your e-mail address"    # fill in the e-mail address of your Copernicus Data Space account
password = getpass.getpass("Password: ")    # the notebook asks for your password when you run this cell

# find the product by its name
answer = requests.get("https://catalogue.dataspace.copernicus.eu/odata/v1/Products",
                      params={"$filter": f"Name eq '{name}'"})
found = answer.json()["value"][0]
product_id = found["Id"]
size = found["ContentLength"]
print("Product found,", round(size / 1e6), "MB")

# log in
login = requests.post("https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
                      data={"client_id": "cdse-public", "grant_type": "password",
                            "username": username, "password": password})
session = requests.Session()
session.headers["Authorization"] = "Bearer " + login.json()["access_token"]
print("Logged in")

# download the zip file, the server redirects us and we follow by hand so that the login is kept
url = f"https://download.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"
print("Asking for the file, it can take a minute before the first data arrives")
response = session.get(url, allow_redirects=False, stream=True)
while response.status_code in (301, 302, 303, 307):
    response = session.get(response.headers["Location"], allow_redirects=False, stream=True)
response.raise_for_status()
print("The file is arriving")

done = 0
with open("product.zip", "wb") as file:
    for chunk in response.iter_content(1 << 20):
        file.write(chunk)
        done += len(chunk)
        print(round(done / 1e6), "MB of", round(size / 1e6), "MB", end="\r")

# unpack it
zipfile.ZipFile("product.zip").extractall(".")
```
:::

The code finds the product by its name, logs in, downloads the zip file and unpacks it. While the file arrives, the cell prints how many megabytes have come so far. The next cell looks at the prepared folder.

In [ ]:
import glob

data_folder = "/scratch/project_2020353/data/exercise_2"    # where the course data is kept on CSC
scene = os.path.join(data_folder, name)
granule = glob.glob(f"{scene}/GRANULE/*")[0]

print(sorted(os.listdir(scene)))
print(sorted(os.listdir(f"{granule}/IMG_DATA")))

for file in sorted(os.listdir(f"{granule}/IMG_DATA/R10m")):
    print(file, round(os.path.getsize(f"{granule}/IMG_DATA/R10m/{file}") / 1e6), "MB")

:::{admonition} Discussion
Compare this folder with the layout printed in Exercise 1. Which files are the bands and which one describes the product? Why are the bands sorted into folders called R10m, R20m and R60m?
:::


### The whole tile at a glance

A tile is a square of 110 km, and every 10 m band has 121 million pixels. For a first look we do not need every pixel. When rasterio reads a band with `out_shape`, it hands us a smaller version, here a tenth of the width and height, and it is quick because the file stores smaller versions of itself. The next cell reads the red, green and blue bands that way and stacks them into a true colour picture of the whole tile.

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from rasterio.enums import Resampling

thumbs = []
for band_name in ["B04", "B03", "B02"]:
    file = glob.glob(f"{granule}/IMG_DATA/R10m/*_{band_name}_10m.jp2")[0]
    with rasterio.open(file) as src:
        thumbs.append(src.read(1, out_shape=(src.height // 10, src.width // 10), resampling=Resampling.average))
        crs = src.crs
        bounds = src.bounds

tile_rgb = (np.dstack(thumbs).astype("float") - 1000) / 10000
tile_rgb = (tile_rgb / 0.15).clip(0, 1)

plt.figure(figsize=(7, 7))
plt.imshow(tile_rgb, extent=(bounds.left, bounds.right, bounds.bottom, bounds.top))
plt.show()

:::{admonition} Discussion
What can you recognise in the picture? How big is the area?
:::


The Vaasa polygon is in another coordinate system than the tile, so it has to be converted before it can be drawn on top. The next cell does that and draws the outline in red.

In [ ]:
outline = vaasa.to_crs(crs)

plt.figure(figsize=(7, 7))
plt.imshow(tile_rgb, extent=(bounds.left, bounds.right, bounds.bottom, bounds.top))
outline.boundary.plot(ax=plt.gca(), color="red")
plt.show()

:::{admonition} Discussion
Why did the polygon have to be converted before we could draw it? What does the red line do at the bottom edge? How much of Vaasa fits into this tile, and what can you see of the city and the river mud at this scale?
:::


### The red band at full resolution

Now we open the red band itself, with all its pixels. The next cell opens the file and prints its properties.

In [ ]:
import rasterio

red_file = glob.glob(f"{granule}/IMG_DATA/R10m/*_B04_10m.jp2")[0]
src = rasterio.open(red_file)

print("Coordinate system", src.crs)
print("Pixel size in metres", src.res)
print("Width and height in pixels", src.width, src.height)
print("Number of pixels", src.width * src.height)

:::{admonition} Discussion
How many pixels does the band have?
:::


### Clipping to Vaasa

Now we clip the tile with the polygon. The polygon is first converted to the coordinate system of the image. Then `mask` keeps only the pixels inside it, and `crop=True` removes the empty rows and columns around it.

In [ ]:
from rasterio.mask import mask

shape = vaasa.to_crs(src.crs).make_valid().iloc[0]
red, transform = mask(src, [shape], crop=True)

pixels_tile = src.width * src.height
pixels_array = red.shape[1] * red.shape[2]
pixels_inside = (red > 0).sum()

print("Pixels in the whole tile", pixels_tile)
print("Pixels in the clipped array", pixels_array, "which is", round(pixels_array / pixels_tile * 100, 1), "% of the tile")
print("Pixels inside the polygon", pixels_inside, "which is", round(pixels_inside / pixels_tile * 100, 1), "% of the tile")
print("Shape of the clipped array (bands, rows, columns)", red.shape)

:::{admonition} Discussion
How many pixels are left, and what share of the tile is that? Why clip the image at all? Is all of the Vaasa polygon inside this tile?
:::


### One band as numbers

The clipped band is an array of whole numbers, one for each pixel. The next cell prints three blocks of 5 by 5 pixels, one in the open sea west of Vaasa, one in the river plume and one in the city centre.

In [ ]:
band = red[0]

points = {"Sea": (760, 890), "Plume": (3120, 1275), "City": (2970, 917)}
for name, (column, row) in points.items():
    print(name)
    print(band[row:row + 5, column:column + 5])

:::{admonition} Discussion
Which numbers are small and which are large? What does a larger number mean for the light that came back from the ground in the red band? If the labels were hidden, which block would you call the sea?
:::


The next cell draws the whole clipped band as a grey image. Dark means a small number and bright a large number.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.imshow(band, cmap="gray", vmin=1000, vmax=2500)
plt.colorbar()

for name, (column, row) in points.items():
    plt.plot(column, row, "o", markersize=9, markeredgecolor="white", label=name)
plt.legend()
plt.show()

:::{admonition} Discussion
What can you recognise in the picture? Where are the sea, the city, the fields and the forest? Why is the area around the polygon black?
:::


### From numbers to reflectance

The numbers are digital numbers, not yet reflectance. To turn them back into reflectance, we divide by a scale of $10000$. After the atmospheric correction a few pixels, for example very dark water, come out slightly below zero, but whole numbers of this type cannot be negative. So ESA also adds an offset of $1000$ to every value, which we subtract again.

In [ ]:
for line in open(f"{scene}/MTD_MSIL2A.xml"):
    if "BOA_ADD_OFFSET band_id" in line or "BOA_QUANTIFICATION_VALUE" in line:
        print(line.strip())

The reflectance is the digital number plus the offset, divided by the scale.

$$\text{reflectance} = \frac{\text{DN} + \text{offset}}{\text{scale}} = \frac{\text{DN} - 1000}{10000}$$

The next cell does this for the whole band and prints the value of the three pixels from before.

In [ ]:
reflectance = (band.astype("float") - 1000) / 10000

for name, (column, row) in points.items():
    print(name, round(reflectance[row, column] * 100, 1), "%")

:::{admonition} Discussion
Is it plausible that the sea sends back about 1 percent of the red light, the plume 3 percent and the city 14 percent? What reflectance do the black pixels outside the polygon get, and what is wrong with it?
:::


### Stacking bands into a colour picture

A true colour picture is three bands shown together, the red band as red, the green band as green and the blue band as blue. We have the red band already. The next cell reads the green and the blue band from the same folder, clips them out with the same polygon, and stacks the three into one array. The three bands have the same pixel size and the same grid, so they fit on top of each other.

In [ ]:
import numpy as np

green_file = glob.glob(f"{granule}/IMG_DATA/R10m/*_B03_10m.jp2")[0]
blue_file = glob.glob(f"{granule}/IMG_DATA/R10m/*_B02_10m.jp2")[0]

green, _ = mask(rasterio.open(green_file), [shape], crop=True)
blue, _ = mask(rasterio.open(blue_file), [shape], crop=True)

stack = np.dstack([red[0], green[0], blue[0]])
print(stack.shape)

:::{admonition} Discussion
What do the three numbers of the shape mean? Why could the three bands be stacked without any further step?
:::


Let's turn them into a picture. The first line turns the digital numbers into reflectance. The second line shows reflectance from 0 to 0.15 as dark to bright, and everything above 0.15 is cut off at white.

In [ ]:
rgb = (stack.astype("float") - 1000) / 10000
rgb = (rgb / 0.15).clip(0, 1)

plt.figure(figsize=(12, 4))
plt.imshow(rgb)
plt.show()

:::{admonition} Discussion
Do the colours look the way you expect from the Copernicus Browser in Exercise 1? What would happen if the 0.15 were 0.05, or 0.3? Who decides how bright a picture looks?
:::


ESA also delivers a ready made true colour image, the file that ends in `TCI_10m`. The next cell clips it out in the same way and draws it.

In [ ]:
tci_file = glob.glob(f"{granule}/IMG_DATA/R10m/*_TCI_10m.jp2")[0]
tci, _ = mask(rasterio.open(tci_file), [shape], crop=True)
print(tci.dtype, tci.shape)

plt.figure(figsize=(12, 4))
plt.imshow(np.moveaxis(tci, 0, -1))
plt.show()

:::{admonition} Discussion
How does ESA's picture differ from ours? What kind of numbers does this file hold, compared with the bands?
:::


### Zooming in

Unfortunately we dont have any zoom in this notebook, but we can zoom by cutting a window out of the array. The numbers in the square brackets are the first and last row and the first and last column. The next cells draw the city centre and the bay in front of it, 150 rows by 200 columns each.

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(rgb[850:1000, 2900:3100], interpolation="nearest")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(rgb[1140:1290, 3020:3220], interpolation="nearest")
plt.show()

:::{admonition} Discussion
How big is one pixel in these pictures, and how big is the whole window on the ground? What can you recognise, and what would be too small to see?
:::


***
## 3 · Which pixels can we trust, and what do they look like

Not every pixel is a clean measurement of the ground. Clouds hide it, their shadows darken it, and some pixels have no data at all. ESA's processing marks these in the **scene classification**, called SCL, a map with one number for every pixel. It has a pixel size of 20 m, so from here on we use the 20 m versions of the bands, from the folder `R20m`. The next cell clips the scene classification out with the same polygon.

In [ ]:
scl_file = glob.glob(f"{granule}/IMG_DATA/R20m/*_SCL_20m.jp2")[0]
scl, _ = mask(rasterio.open(scl_file), [shape], crop=True)
scl = scl[0]
print(scl.shape)

The numbers in the scene classification mean the following. The map in the next cells uses the colours in the last column. The classification is made by ESA's processor from the measured bands, it is not a measurement itself, and it can be wrong. The full description is on [SentiWiki](https://sentiwiki.copernicus.eu/web/s2-products).

:::{list-table}
:header-rows: 1
:widths: 10 50 40
:align: left

* - Number
  - Class
  - Colour
* - 0
  - no data
  - <span style="display:inline-block;width:1.1em;height:0.8em;background:black;border:1px solid #888;vertical-align:middle;margin-right:0.5em"></span>black
* - 1
  - saturated or defective
  - <span style="display:inline-block;width:1.1em;height:0.8em;background:red;border:1px solid #888;vertical-align:middle;margin-right:0.5em"></span>red
* - 2
  - dark area pixels
  - <span style="display:inline-block;width:1.1em;height:0.8em;background:grey;border:1px solid #888;vertical-align:middle;margin-right:0.5em"></span>grey
* - 3
  - cloud shadows
  - <span style="display:inline-block;width:1.1em;height:0.8em;background:brown;border:1px solid #888;vertical-align:middle;margin-right:0.5em"></span>brown
* - 4
  - vegetation
  - <span style="display:inline-block;width:1.1em;height:0.8em;background:green;border:1px solid #888;vertical-align:middle;margin-right:0.5em"></span>green
* - 5
  - not vegetated
  - <span style="display:inline-block;width:1.1em;height:0.8em;background:yellow;border:1px solid #888;vertical-align:middle;margin-right:0.5em"></span>yellow
* - 6
  - water
  - <span style="display:inline-block;width:1.1em;height:0.8em;background:blue;border:1px solid #888;vertical-align:middle;margin-right:0.5em"></span>blue
* - 7
  - unclassified
  - <span style="display:inline-block;width:1.1em;height:0.8em;background:magenta;border:1px solid #888;vertical-align:middle;margin-right:0.5em"></span>magenta
* - 8
  - cloud, medium probability
  - <span style="display:inline-block;width:1.1em;height:0.8em;background:lightgrey;border:1px solid #888;vertical-align:middle;margin-right:0.5em"></span>light grey
* - 9
  - cloud, high probability
  - <span style="display:inline-block;width:1.1em;height:0.8em;background:white;border:1px solid #888;vertical-align:middle;margin-right:0.5em"></span>white
* - 10
  - thin cirrus
  - <span style="display:inline-block;width:1.1em;height:0.8em;background:lightblue;border:1px solid #888;vertical-align:middle;margin-right:0.5em"></span>light blue
* - 11
  - snow or ice
  - <span style="display:inline-block;width:1.1em;height:0.8em;background:cyan;border:1px solid #888;vertical-align:middle;margin-right:0.5em"></span>cyan
:::

In [ ]:
import pandas as pd

inside = scl[scl > 0]
share = pd.Series(inside).value_counts(normalize=True).sort_index() * 100
print(share.round(2))

:::{admonition} Discussion
Which classes are the biggest? How much of Vaasa is cloud or cloud shadow? Is that as clear as the catalogue table told us for this date?
:::


In [ ]:
from matplotlib.colors import ListedColormap

colours = ["black", "red", "grey", "brown", "green", "yellow", "blue", "magenta", "lightgrey", "white", "lightblue", "cyan"]
scl_colours = ListedColormap(colours)

plt.figure(figsize=(12, 4))
plt.imshow(scl, cmap=scl_colours, vmin=0, vmax=11, interpolation="nearest")
plt.show()

Now one cloud, close up. The next cell finds the middle of all the pixels that are classified as cloud and draws a window of 3 by 3 km around it, the true colour picture on the left and the scene classification on the right.

In [ ]:
# Find the cloud pixels, class 8 is medium and class 9 is high probability cloud
cloud_rows, cloud_columns = np.where(np.isin(scl, [8, 9]))

# Take the middle of the cloud pixels as the centre of the window, in 20 m pixels
centre_row = int(np.median(cloud_rows))
centre_column = int(np.median(cloud_columns))

# The window is 150 by 150 pixels in the 20 m classification
half = 75
scl_window = (slice(centre_row - half, centre_row + half),
              slice(centre_column - half, centre_column + half))

# The colour picture has 10 m pixels, so the same place has twice the row and column numbers
rgb_window = (slice(2 * (centre_row - half), 2 * (centre_row + half)),
              slice(2 * (centre_column - half), 2 * (centre_column + half)))

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(rgb[rgb_window])
plt.title("True colour")

plt.subplot(1, 2, 2)
plt.imshow(scl[scl_window], cmap=scl_colours, vmin=0, vmax=11, interpolation="nearest")
plt.title("Scene classification")

plt.show()

:::{admonition} Discussion
What do you see in the true colour picture, and what does the classification make of it? Where is the shadow in relation to the cloud, and what does that tell you about where the sun was?
:::


### Spectral signatures by class

The classification gives us a way to ask what each kind of pixel looks like in every band. The next cell reads nine bands of 20 m, converts them to reflectance and stacks them. The cell after it calculates the mean reflectance of five groups of pixels in each band and draws one line for each group. This is a spectral signature.

In [ ]:
bands = ["B02", "B03", "B04", "B05", "B06", "B07", "B8A", "B11", "B12"]
wavelength = [490, 560, 665, 705, 740, 783, 865, 1610, 2190]

cube = []
for band_name in bands:
    file = glob.glob(f"{granule}/IMG_DATA/R20m/*_{band_name}_20m.jp2")[0]
    data, _ = mask(rasterio.open(file), [shape], crop=True)
    cube.append((data[0].astype("float") - 1000) / 10000)
cube = np.array(cube)
print(cube.shape)

In [ ]:
groups = {"water": [6], "vegetation": [4], "not vegetated": [5], "cloud": [8, 9], "cloud shadow": [3]}

for name, classes in groups.items():
    pixels = np.isin(scl, classes)
    plt.plot(wavelength, [band[pixels].mean() for band in cube], marker="o", label=name)
plt.xlabel("wavelength (nm)")
plt.ylabel("reflectance")
plt.legend()
plt.show()

:::{admonition} Discussion
Which line is which? What does vegetation do between the red and the near infrared? What is special about the cloud, and which two lines are hard to tell apart?
:::


### Filtering, before and after

Now we remove the pixels we cannot trust, the ones classified as saturated or defective, cloud shadow, cloud and thin cirrus, and compare the numbers before and after. The next cell does this for all of Vaasa.

In [ ]:
inside = scl > 0
bad = np.isin(scl, [1, 3, 8, 9, 10])
usable = inside & ~bad

print("Pixels inside Vaasa", inside.sum())
print("Pixels we remove", bad.sum(), "which is", round(bad.sum() / inside.sum() * 100, 2), "%")

for name, values in [("Blue", cube[0]), ("Green", cube[1]), ("Red", cube[2])]:
    before = values[inside].mean()
    after = values[usable].mean()
    print(name, "mean before", round(before, 4), "after", round(after, 4), "change", round((after - before) / before * 100, 1), "%")

The same numbers, only for the window around the cloud that we looked at.

In [ ]:
inside_box = inside[box]
usable_box = usable[box]

print("Pixels inside the window", inside_box.sum())
print("Pixels removed", inside_box.sum() - usable_box.sum())

before = cube[2][box][inside_box].mean()
after = cube[2][box][usable_box].mean()
print("Red mean before", round(before, 4), "after", round(after, 4), "change", round((after - before) / before * 100, 1), "%")

:::{admonition} Discussion
How many pixels did we remove in Vaasa as a whole, and how much did the means change? What happened near the cloud? What would happen on a day with a lot of cloud, and what does this mean for a turbidity map?
:::


### Other methods, not shown today

Filtering with the scene classification is the simplest way to deal with clouds. There are others. These pages are for the assignment and for later.

- **Cloud probability masks.** The `s2cloudless` detector gives a cloud probability for each pixel, and you choose the threshold yourself. See [sentinel2-cloud-detector](https://github.com/sentinel-hub/sentinel2-cloud-detector). The coastal turbidity notebook of Digital Earth Australia uses it together with a water mask, see [Visualising coastal turbidity with Sentinel-2](https://knowledge.dea.ga.gov.au/notebooks/Real_world_examples/Turbidity_animated_timeseries).
- **Joining tiles.** Vaasa lies on the edge of the tile, so a full map needs two or more tiles joined. See [rasterio.merge](https://rasterio.readthedocs.io/en/stable/api/rasterio.merge.html).
- **Cloud free composites.** Take many dates, remove the bad pixels of each, and take the median per pixel. See the [stackstac basics](https://stackstac.readthedocs.io/en/latest/basic.html) and the [Planetary Computer Sentinel-2 example](https://nbviewer.org/github/microsoft/PlanetaryComputerExamples/blob/main/datasets/sentinel-2-l2a/sentinel-2-l2a-example.ipynb).
- **Downloading with other tools.** One interface for several data providers, see the [EODAG tutorials](https://eodag.readthedocs.io/en/stable/tutos.html). The Finnish open data portal [Paituli](https://paituli.csc.fi/geoserver/ogc/stac/v1) offers Sentinel-2 and ready-made monthly index mosaics through the same kind of catalogue search that we used.